# Install instructions

1. Clone repository 
```bash
git clone https://github.com/cda-tum/mqt-qmap/tree/na-zx-mapping-synthesis --recursive
```
2. create venv and inside directory
```bash
pip install .
```

# Use case example


In [1]:
from qiskit import QuantumCircuit

from mqt.qmap import HybridSynthesisMapper, NeutralAtomHybridArchitecture, HybridMapperParameters, InitialCoordinateMapping, InitialCircuitMapping

In [2]:
# create two possible synthesis steps
synthesis_step_1 = QuantumCircuit(3)
synthesis_step_1.cx(0, 1)
synthesis_step_1.cx(0, 2)

synthesis_step_2 = QuantumCircuit(3)
synthesis_step_2.cx(0, 2)
synthesis_step_2.cx(1, 2)

synthesis_stesps = [synthesis_step_1, synthesis_step_2]


## Create Mapper

In [3]:
# create a neutral atom hybrid architecture
arch_file = 'rubidium.json'
architecture = NeutralAtomHybridArchitecture(arch_file)

# set mapper parameters (skip to use default values)
params = HybridMapperParameters()
# mapper should use SWAP gates or shuttling operations
params.gate_weight = 0
params.shuttling_weight = 1
# look-ahead weights
params.lookahead_weight_moves = 0.1
params.lookahead_weight_swaps = 0.1
# The initial mapping between atoms and hardware
params.initial_mapping = InitialCoordinateMapping.trivial
# If mapper should print debug information
params.verbose = True

# create mapper
synthesis_mapper = HybridSynthesisMapper(arch=architecture, params=params)
# alternatively
# synthesis_mapper = HybridSynthesisMapper(arch=architecture)
# synthesis_mapper.set_parameters(params)


## Initialize mapping process


In [4]:
# set the initial circuit mapping and the size of the mapping(number of qubits)
synthesis_mapper.init_mapping(n_qubits=3, initial_mapping=InitialCircuitMapping.identity)

## Evaluate different synthesis steps

In [5]:
# pass the synthesis steps to the mapper
# the returned index is the index of the synthesis step that is the best fit for the architecture
# the parameter indicates if the step should be directly applied or not 
index = synthesis_mapper.evaluate_synthesis_steps(synthesis_stesps, also_map=False)
print(index)

1
Evaluating synthesis step number 0
mapped h 1 
mapped h 2 
mapped z 0 1 
iteration 1
moved 1 to 13  logical qubit: 1
iteration 2
moved 2 to 1  logical qubit: 2
mapped h 1 
mapped z 0 2 
f,g: f,s: l,g: 
l,g: 
nSwaps: 0
nMoves: 2
nMoveGroups: 2
Fidelity: 0.989568
Evaluating synthesis step number 2
mapped h 2 
mapped z 1 2 
iteration 1
moved 1 to 13  logical qubit: 1
iteration 2
moved 2 to 1  logical qubit: 2
mapped z 0 2 
f,g: f,s: l,g: 
l,g: 
nSwaps: 0
nMoves: 2
nMoveGroups: 2
Fidelity: 0.990558


## Other methods

In [6]:
# reset the mapper
synthesis_mapper.init_mapping(n_qubits=3, initial_mapping=InitialCircuitMapping.identity)

In [7]:
# append a circuit to the mapper by mapping it to the architecture and adding it to the circuit
synthesis_mapper.append_with_mapping(synthesis_step_1)

mapped h 1 
mapped h 2 
mapped z 0 1 
iteration 1
moved 1 to 13  logical qubit: 1
iteration 2
moved 2 to 1  logical qubit: 2
mapped h 1 
mapped z 0 2 
f,g: f,s: l,g: 
l,g: 
nSwaps: 0
nMoves: 2


In [8]:
# pass a circuit to the mapper without mapping it to the architecture
# the circuit needs to fit the architecture!!!
synthesis_mapper.append_without_mapping(synthesis_step_2)

mapped x 0 2 
mapped x 1 2 


In [9]:
# remap the whole circuit again from scratch
synthesis_mapper.complete_remap()

mapped h 1 
mapped h 2 
mapped z 0 2 
mapped z 0 2 
iteration 1
moved 1 to 2  logical qubit: 2
iteration 2
moved 13 to 1  logical qubit: 1
mapped z 0 1 
mapped h 1 
mapped z 1 2 
f,g: f,s: l,g: 
l,g: 
nSwaps: 0
nMoves: 2


## Get resulting circuits

In [10]:
# the complete synthesized circuit
synthesized_circuit_qasm = synthesis_mapper.get_synthesized_qc()
synthesized_circuit = QuantumCircuit.from_qasm_str(synthesized_circuit_qasm)
synthesized_circuit.draw()

q_0: ──────■───────■──■─────────
     ┌───┐ │ ┌───┐ │  │         
q_1: ┤ H ├─■─┤ H ├─┼──┼──■──────
     ├───┤   └───┘ │  │  │ ┌───┐
q_2: ┤ H ├─────────■──■──■─┤ H ├
     └───┘                 └───┘

In [11]:
# the mapped circuit
mapped_circuit_qasm = synthesis_mapper.get_mapped_qc()
# can not be drawn as it is not valid qasm (move operations are not supported by qiskit)
print(mapped_circuit_qasm)

// i 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15
// o 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15
OPENQASM 2.0;
include "qelib1.inc";
qreg q[16];
h q[13];
h q[1];
cz q[0], q[1];
cz q[0], q[1];
move q[1], q[2];
move q[13], q[1];
cz q[0], q[1];
h q[1];
cz q[1], q[2];



In [12]:
# convert moves to aod operations
synthesis_mapper.convert_to_aod()
mapped_circuit_qasm = synthesis_mapper.get_mapped_qc_aod()
print(mapped_circuit_qasm)

// i 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15
// o 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15
OPENQASM 2.0;
include "qelib1.inc";
qreg q[16];
h q[13];
h q[1];
cz q[0], q[1];
cz q[0], q[1];
aod_activate (0, 3, 3; 1, 0, 0;) q[1];
aod_move (0, 3, 3.1; 1, 0, 0.1;) q[1], q[2];
aod_move (0, 3.1, 5.9;) q[1], q[2];
aod_move (0, 5.9, 6; 1, 0.1, 0;) q[1], q[2];
aod_deactivate (0, 6, 6; 1, 0, 0;) q[2];
aod_activate (0, 3, 3; 1, 9, 9;) q[13];
aod_move (0, 3, 3.1; 1, 9, 8.9;) q[13], q[1];
aod_move (1, 8.9, 0.1;) q[1], q[13];
aod_move (0, 3.1, 3; 1, 0.1, 0;) q[13], q[1];
aod_deactivate (0, 3, 3; 1, 0, 0;) q[1];
cz q[0], q[1];
h q[1];
cz q[1], q[2];

nMoveGroups: 2


## Get connectivity

In [12]:
import numpy as np

In [13]:
# get adjacency matrix of the architecture at the current state of the mapper
adjacency_matrix = np.array(synthesis_mapper.get_circuit_adjacency_matrix())
print(adjacency_matrix)

[[0 1 0]
 [1 0 1]
 [0 1 0]]
